# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [2]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Detected BASE: /Users/guptam8/Downloads/FlashEats_Classroom_Pack_V2
Database path: /Users/guptam8/Downloads/FlashEats_Classroom_Pack_V2/database/flasheats.db
Database exists: True


## Load the updated FlashEats data

In [3]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?

Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique in orders table | `orders['order_id'].duplicated().sum() == 0` | High (distorts denominator) |
| Delivered orders have completion time | `actual_delivery_at` is non-null for delivered orders | Count nulls where `final_status == 'delivered'` | High (cannot calculate late rate) |
| Promised ETA is valid | `promised_eta >= created_at` | Count rows where `promised_eta < created_at` | Medium (chronology defect) |
| Event chronology is valid | `pickup_at <= actual_delivery_at` | Count rows where `pickup_at > actual_delivery_at` | High (physically impossible) |
| “Late” has an agreed definition | Single documented threshold owned by business | Compare VP Ops, Support, Finance definitions in `client_metric_definitions.json` | High (misleading reporting) |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*
- Duplicate rows inflate the denominator or numerator.
- 37 delivered orders lack actual delivery timestamps.
- Disagreement on whether minor delays (< 10 min) should count as late.


In [4]:
# Challenge 1: Test validation contract assumptions
print('Total rows:', len(orders))
print('Unique order IDs:', orders['order_id'].nunique())
print('Duplicate order IDs count:', orders['order_id'].duplicated().sum())

# Deduplicate orders for analysis
orders_clean = orders.drop_duplicates(subset=['order_id'], keep='first').copy()

# Parse datetime columns
for c in ['created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at']:
    orders_clean[c] = pd.to_datetime(orders_clean[c], format='mixed', errors='coerce')

# 1. Delivered orders with missing completion timestamp
delivered_mask = orders_clean['final_status'].str.lower() == 'delivered'
missing_delivery = orders_clean[delivered_mask]['actual_delivery_at'].isna().sum()
print(f'Delivered orders with missing completion timestamp: {missing_delivery}')

# 2. Test promised_eta >= created_at
bad_eta = (orders_clean['promised_eta'] < orders_clean['created_at']).sum()
print(f'Orders where promised_eta < created_at: {bad_eta}')

# 3. Test pickup_at <= actual_delivery_at
bad_pickup = (orders_clean['pickup_at'] > orders_clean['actual_delivery_at']).sum()
print(f'Orders where pickup_at > actual_delivery_at: {bad_pickup}')

# 4. Summary of issues requiring clarification
print('\nAssumptions needing stakeholder clarification:')
print('- 3 duplicate order rows need deduplication rules.')
print('- 37 delivered orders with missing timestamps: count as on-time or exclude?')
print('- 5 orders with pickup after delivery: corrupted sensor data to discard.')


Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


# Challenge 2 — Stakeholders disagree on “late”

Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|
| VP Operations (delay > 0 min) | **56.4%** (843 / 1,495) | Strict SLA: Any delivered order arriving after promised ETA |
| Support Lead (delay > 10 min) | **23.3%** (349 / 1,495) | Meaningfully late: Delays that provoke customer support complaints |
| Finance (all placed orders) | **52.7%** (843 / 1,600) | Overall order base: Excludes cancellations from delivered SLA |
| Historical dashboard (assume missing = on time) | **55.0%** (843 / 1,532) | Total delivered orders assuming missing timestamps were on time |

### Decision on publication and ownership
> **Which one should leadership publish, and who must own that decision?**

**Answer:**
Leadership should **NOT** publish the '56% Late Delivery Rate' today. Publishing 56% conflates a 30-second delay with a 45-minute delay and hides the fact that only 23.3% of orders are late enough to cause customer dissatisfaction (>10 min).

The **VP of Operations** and **Head of Product/Analytics** must co-own the metric definition. They should establish a two-tiered KPI:
1. **Operational SLA Rate** (delay > 0 min) for internal driver dispatching and kitchen tracking.
2. **Customer Late Rate** (delay > 10 min) for leadership reporting and executive KPI tracking.


In [5]:
# Challenge 2: Calculate late rate under different stakeholder definitions
analysis = orders.drop_duplicates('order_id', keep='first').copy()

for c in ['promised_eta', 'actual_delivery_at']:
    analysis[c] = pd.to_datetime(analysis[c], format='mixed', errors='coerce')

analysis['delay_min'] = (
    analysis['actual_delivery_at'] - analysis['promised_eta']
).dt.total_seconds() / 60

delivered_valid = analysis[(analysis['final_status'].str.lower() == 'delivered') & (analysis['actual_delivery_at'].notna())]
delivered_all = analysis[analysis['final_status'].str.lower() == 'delivered']

# 1. VP Operations: Any delay > 0 min among delivered with timestamps
late_vp = (delivered_valid['delay_min'] > 0).sum()
rate_vp = late_vp / len(delivered_valid)

# 2. Support Lead: Delay > 10 min
late_support = (delivered_valid['delay_min'] > 10).sum()
rate_support = late_support / len(delivered_valid)

# 3. Total unique orders denominator
rate_total = late_vp / len(analysis)

# 4. All delivered including null actuals
rate_all_delivered = late_vp / len(delivered_all)

print(f'VP Operations (delay > 0 min): {late_vp}/{len(delivered_valid)} = {rate_vp:.1%}')
print(f'Support Lead (delay > 10 min): {late_support}/{len(delivered_valid)} = {rate_support:.1%}')
print(f'Total order base (delay > 0 / 1600): {late_vp}/{len(analysis)} = {rate_total:.1%}')
print(f'All delivered base (delay > 0 / 1532): {late_vp}/{len(delivered_all)} = {rate_all_delivered:.1%}')


# Challenge 3 — Validate categories without cleaning by instinct

Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

### Categorical Audit

| Column | Observed Values | Normalization Rule | Owner Confirmation Required? |
|---|---|---|---|
| `orders.final_status` | `delivered` (1530), `cancelled` (68), `Delivered` (5) | Convert to lowercase (`.str.lower()`) | No (clean representation difference) |
| `orders.traffic_bucket` | `medium`, `high`, `low`, `severe`, `HIGH` | Convert to lowercase (`.str.lower()`) | No (clean representation difference) |
| `restaurant_status.status` | `preparing`, `handed_off`, `ready`, `ready `, `READY`, `Ready`, `handoff`, `unknown` | Strip whitespace, lowercase; map `handoff` to `handed_off` | Yes — confirm if `handoff` equals `handed_off` and investigate `unknown` |
| `tickets.category` | `late_delivery`, `eta_changed`, `restaurant_delay`, `ready_but_waiting`, `status_mismatch`, `driver_not_moving`, `Late Delivery`, `late_delivery `, `ETA issue` | Strip whitespace, lowercase | Yes — confirm with Support if `ETA issue` maps to `eta_changed` |


In [6]:
# Challenge 3: Inspect categorical distributions
for col in ['final_status', 'traffic_bucket']:
    print(f'\n--- orders.{col} ---')
    print(orders[col].value_counts(dropna=False))

print('\n--- restaurant_status.status ---')
print(restaurant_status['status'].value_counts(dropna=False))

print('\n--- tickets.category ---')
print(tickets['category'].value_counts(dropna=False))

# Demonstration of safe normalization
orders_clean['final_status_clean'] = orders_clean['final_status'].astype(str).str.strip().str.lower()
orders_clean['traffic_bucket_clean'] = orders_clean['traffic_bucket'].astype(str).str.strip().str.lower()
restaurant_status['status_clean'] = restaurant_status['status'].astype(str).str.strip().str.lower().replace({'handoff': 'handed_off'})
tickets['category_clean'] = tickets['category'].astype(str).str.strip().str.lower().replace({'late delivery': 'late_delivery', 'eta issue': 'eta_changed'})

print('\nCleaned restaurant statuses:')
print(restaurant_status['status_clean'].value_counts())



 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


# Challenge 4 — Cross-source integrity

Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|
| `orders.restaurant_id` → `restaurants` | 100.0% | PASS | Low — all restaurant IDs resolve cleanly |
| `orders.driver_id` → `drivers` | 100.0% | PASS | Low — all assigned drivers exist in drivers table |
| `tickets.order_id` → `orders` | 100.0% | PASS | Low — all tickets belong to valid orders |
| `restaurant_status.order_id` → `orders` | 100.0% | PASS | Low — all status updates correspond to valid orders |

> **If 1% is unmapped, is that acceptable?**

**Answer:**
It depends on the business decision:
- For **weekly executive KPI tracking**, 99% coverage is acceptable because 1% will not distort aggregate trends.
- For **driver earnings or restaurant billing**, 1% unmapped is **UNACCEPTABLE** because it results in missing payouts, billing disputes, and legal non-compliance.


In [7]:
# Challenge 4: Calculate cross-source mapping coverage
cov_restaurant = orders['restaurant_id'].dropna().isin(restaurants['restaurant_id']).mean() * 100
cov_driver = orders['driver_id'].dropna().isin(drivers['driver_id']).mean() * 100
cov_tickets = tickets['order_id'].dropna().isin(orders['order_id']).mean() * 100
cov_status = restaurant_status['order_id'].dropna().isin(orders['order_id']).mean() * 100

print(f'orders.restaurant_id -> restaurants: {cov_restaurant:.2f}% coverage')
print(f'orders.driver_id -> drivers:         {cov_driver:.2f}% coverage')
print(f'tickets.order_id -> orders:          {cov_tickets:.2f}% coverage')
print(f'restaurant_status.order_id -> orders: {cov_status:.2f}% coverage')


# Challenge 5 — Freshness is an SLA question

Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

| Use Case | SLA Status | Reason |
|---|---|---|
| **Weekly Analytics** | **PASS** | Batch analysis operates on historical records; all updates are available after order completion. |
| **Live Customer ETA** | **FAIL** | 167 updates occur after driver pickup, and some update timestamps precede order creation. Live ETA would display stale data. |
| **Restaurant Accountability** | **FAIL** | Status timestamps do not accurately measure kitchen ready times; using them to penalize restaurants would cause disputes. |


In [8]:
# Challenge 5: Merge restaurant_status with orders and analyze timing freshness
merged_status = restaurant_status.merge(orders_clean, on='order_id', how='inner')
for c in ['last_updated_at', 'created_at', 'pickup_at', 'actual_delivery_at']:
    merged_status[c] = pd.to_datetime(merged_status[c], format='mixed', errors='coerce')

merged_status['delta_created_min'] = (merged_status['last_updated_at'] - merged_status['created_at']).dt.total_seconds() / 60
merged_status['delta_pickup_min'] = (merged_status['last_updated_at'] - merged_status['pickup_at']).dt.total_seconds() / 60

print('Summary of status timestamp vs created_at (min):')
print(merged_status['delta_created_min'].describe().round(1))

print('\nFreshness anomalies:')
print('Status updates timestamped BEFORE order creation:', (merged_status['delta_created_min'] < 0).sum())
print('Status updates timestamped AFTER driver pickup:', (merged_status['delta_pickup_min'] > 0).sum())


# Challenge 6 — Build the validation gate

Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | **WARN** | 3 duplicate order IDs (6 rows) | Deduplicate orders by `order_id` keeping first record |
| Timestamp chronology | **WARN** | 4 orders with ETA < created, 5 with pickup > delivery | Filter corrupt timestamps before computing transit metrics |
| KPI definition | **FAIL** | Disagreement: VP Ops (56.4%) vs Support (23.3%) | Require executive committee to formally define KPI threshold |
| Category semantics | **WARN** | Inconsistent casing (`Delivered`, `HIGH`) and ambiguous status (`handoff`) | Normalize casing and confirm semantics with system owners |
| Cross-source mapping | **PASS** | 100.0% coverage across orders, drivers, restaurants, tickets | Accept without modification |
| Freshness | **FAIL** | Status timestamps lag pickup and precede order creation | Block use for live ETA; allow only for batch analytics |

### Final Publishing Decision
> **Should leadership publish “Late Delivery Rate = 56%” today?**

**NO.** Leadership must not publish this number today.

**What exactly must happen before the metric becomes publishable?**
1. **Formal KPI Signoff:** Executive leadership must formally choose between strict SLA (> 0 min = 56.4%) and customer-perceived lateness (> 10 min = 23.3%).
2. **Deduplication Policy:** Enforce ingestion-level deduplication so duplicate order records do not skew counts.
3. **Missing Data Policy:** Decide whether 37 delivered orders with missing delivery times are excluded or imputed.


In [9]:
# Challenge 6: Populate validation report gate
validation_report = {
    'business_grain': 'WARN',
    'timestamp_chronology': 'WARN',
    'kpi_definition': 'FAIL',
    'category_semantics': 'WARN',
    'cross_source_mapping': 'PASS',
    'freshness': 'FAIL',
    'publish_56_percent': 'NO'
}

print('=== FINAL VALIDATION GATE REPORT ===')
for check, status in validation_report.items():
    print(f'{check:25s}: {status}')


{'business_grain': None, 'timestamp_chronology': None, 'kpi_definition': None, 'category_semantics': None, 'cross_source_mapping': None, 'freshness': None, 'publish_56_percent': None}


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**